In [ ]:
!pip install QuantLib

In [ ]:
import QuantLib as ql
import numpy as np
import pandas as pd
import time

In [ ]:
def american_call_heston(S, K, T, r, q, v0, theta, kappa, sigma, rho):
    '''
    S: Current price of the underlying asset
    K: Strike price of the option
    T: Time to maturity of the option (in years)
    r: Risk-free interest rate
    q: Dividend yield of the underlying asset
    v0: Initial volatility of the Heston model
    theta: Mean reversion level of the Heston model
    kappa: Mean reversion speed of the Heston model
    sigma: Volatility of variance in the Heston model
    rho: Correlation between the asset price and volatility processes
    '''

    # Set up the evaluation date
    today = ql.Date.todaysDate()
    expiry_date = today + ql.Period(int(T * 365), ql.Days)

    # Define the payoff and exercise
    payoff = ql.PlainVanillaPayoff(ql.Option.Call, K)
    exercise = ql.AmericanExercise(today, expiry_date)

    # Create the vanilla option
    option = ql.VanillaOption(payoff, exercise)

    # Set up the market data
    spot = ql.SimpleQuote(S)
    rate = ql.SimpleQuote(r)
    dividend = ql.SimpleQuote(q)

    flat_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, ql.QuoteHandle(rate), ql.Actual365Fixed()))
    dividend_yield = ql.YieldTermStructureHandle(ql.FlatForward(today, ql.QuoteHandle(dividend), ql.Actual365Fixed()))
    spot_handle = ql.QuoteHandle(spot)

    # Set up the Heston model
    heston_process = ql.HestonProcess(flat_ts, dividend_yield, spot_handle, v0, kappa, theta, sigma, rho)
    heston_model = ql.HestonModel(heston_process)

    # Define the FDHestonVanillaEngine and inputs
    tGrid, xGrid, vGrid = 100, 100, 50
    dampingSteps = 0
    fdScheme = ql.FdmSchemeDesc.ModifiedCraigSneyd()
    heston_engine = ql.FdHestonVanillaEngine(heston_model, tGrid, xGrid, vGrid, dampingSteps, fdScheme)
    option.setPricingEngine(heston_engine)

    # Return the calculated option price
    return option.NPV()

In [ ]:
S = 100  # Fixed stock price
N = 20000  # Number of samples

def generate_parameters(N):
    """Generate random parameters for the Heston model."""
    K = np.random.uniform(90, 110, N)
    q = np.random.uniform(0.01, 0.03, N)
    r = np.random.uniform(0.01, 0.06, N)
    v0 = np.random.uniform(0.2, 0.5, N)
    t = np.random.uniform(0.1, 1, N)
    theta = np.random.uniform(0.01, 2.0, N)
    kappa = np.random.uniform(0.01, 2.0, N)
    sigma = np.random.uniform(0.01, 1.0, N)
    rho = np.random.uniform(-0.9, 0.9, N)

    return K, q, r, v0, t, theta, kappa, sigma, rho

K, q, r, v0, t, theta, kappa, sigma, rho = generate_parameters(N) #generate parameters

while True:
    valid_indices = np.where(2 * kappa * theta > sigma**2)[0] #check Feller conditions

    if len(valid_indices) >= N: #check if we have 20000 samples after feller conditions
        break

    # If less than N samples, generate more parameters, add to old valid samples
    new_N = N - len(valid_indices)
    new_params = generate_parameters(new_N)
    K = np.concatenate([K, new_params[0]])
    q = np.concatenate([q, new_params[1]])
    r = np.concatenate([r, new_params[2]])
    v0 = np.concatenate([v0, new_params[3]])
    t = np.concatenate([t, new_params[4]])
    theta = np.concatenate([theta, new_params[5]])
    kappa = np.concatenate([kappa, new_params[6]])
    sigma = np.concatenate([sigma, new_params[7]])
    rho = np.concatenate([rho, new_params[8]])

# Create parameter list of N samples for each parameter
K = K[valid_indices]
q = q[valid_indices]
r = r[valid_indices]
v0 = v0[valid_indices]
t = t[valid_indices]
theta = theta[valid_indices]
kappa = kappa[valid_indices]
sigma = sigma[valid_indices]
rho = rho[valid_indices]

GeneratePricesStart = time.time()
# Calculate Option prices
option_prices = []
for i in range(N):
    try:
        price = american_call_heston(
            S=S,
            K=K[i],
            T=t[i],
            r=r[i],
            q=q[i],
            v0=v0[i],
            theta=theta[i],
            kappa=kappa[i],
            sigma=sigma[i],
            rho=rho[i]
        )
        option_prices.append(price) #add price to list
    except RuntimeError as e: #check for error
        option_prices.append(np.nan) #if error, price = nan
    if (i + 1) % 1000 == 0:
        print(f'{i + 1} iterations completed') #keep track of how many iterations completed in price generation

#code made in collaboration with ChatGPT
# While loop to recalculate error parameters (nan) and if price value does not make sense (price less than 0)
while np.isnan(option_prices).any() or any(price < 0 for price in option_prices):

    invalid_count = np.sum(np.isnan(option_prices) | (np.array(option_prices) < 0)) #count # of errors
    invalid_indices = np.isnan(option_prices) | (np.array(option_prices) < 0) #keep track of error indices

    # Generate new random values for the invalid samples
    new_K = np.random.uniform(90, 110, invalid_count)
    new_q = np.random.uniform(0.01, 0.03, invalid_count)
    new_r = np.random.uniform(0.01, 0.06, invalid_count)
    new_v0 = np.random.uniform(0.2, 0.5, invalid_count)
    new_t = np.random.uniform(0.1, 1, invalid_count)
    new_theta = np.random.uniform(0.01, 2.0, invalid_count)
    new_kappa = np.random.uniform(0.01, 2.0, invalid_count)
    new_sigma = np.random.uniform(0.01, 1.0, invalid_count)
    new_rho = np.random.uniform(-0.9, 0.9, invalid_count)

    # Replace errors with new samples
    K[invalid_indices] = new_K
    q[invalid_indices] = new_q
    r[invalid_indices] = new_r
    v0[invalid_indices] = new_v0
    t[invalid_indices] = new_t
    theta[invalid_indices] = new_theta
    kappa[invalid_indices] = new_kappa
    sigma[invalid_indices] = new_sigma
    rho[invalid_indices] = new_rho

    # Re-run the calculations for errors
    for i in range(N):
        if np.isnan(option_prices[i]) or option_prices[i] < 0 or option_prices[i] > 100:  # Only recalculate NaNs
            try:
                price = american_call_heston(
                    S=S,
                    K=K[i],
                    T=t[i],
                    r=r[i],
                    q=q[i],
                    v0=v0[i],
                    theta=theta[i],
                    kappa=kappa[i],
                    sigma=sigma[i],
                    rho=rho[i]
                )
                option_prices[i] = price  # Update the option price
            except RuntimeError as e:
                option_prices[i] = np.nan
GeneratePricesEnd = time.time()

print(f'Price generation took {GeneratePricesEnd - GeneratePricesStart} seconds')

In [ ]:
#save prices to dataframe
data = {
        'Strike Price (K)': K,
        'Dividend Yield (q)': q,
        'Interest Rate (r)': r,
        'Volatility (v0)': v0,
        'Time to Maturity (T)': t,
        'Theta': theta,
        'Kappa': kappa,
        'Sigma': sigma,
        'Rho': rho,
        'Call Option Value (C)': option_prices
    }
df = pd.DataFrame(data)